In [ ]:
from __future__ import annotations

from pathlib import Path
import pickle

import numpy as np

import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from scipy.linalg import subspace_angles
from scipy.stats import ttest_ind

import manifold_dynamics.neural_utils as nu
import manifold_dynamics.paths as pth
import manifold_dynamics.tuning_utils as tut
import visionlab_utils.storage as vst

def diagonal_profile(M):
    n = M.shape[0]
    vals = []
    for k in range(1, n):
        vals.append(np.mean(np.diag(M, k)))
    return np.array(vals)

def change_point_score(M, min_block=5):
    n = M.shape[0]
    scores = np.full(n, np.nan)

    for s in range(min_block, n - min_block):
        ee = M[:s, :s]
        ll = M[s:, s:]
        el = M[:s, s:]

        # exclude diagonals from within-block means
        ee_mask = ~np.eye(ee.shape[0], dtype=bool)
        ll_mask = ~np.eye(ll.shape[0], dtype=bool)

        ee_mean = ee[ee_mask].mean() if ee.shape[0] > 1 else np.nan
        ll_mean = ll[ll_mask].mean() if ll.shape[0] > 1 else np.nan
        el_mean = el.mean()

        scores[s] = el_mean - 0.5 * (ee_mean + ll_mean)

    best_split = np.nanargmax(scores)
    return best_split, scores

In [ ]:
# ROI name, should be 3 parts (index.label.category)
target = "08.MF1.F"
target_parts = target.split(".")
roi_label = f"{int(target_parts[0]):02d}.{target_parts[1]}.{target_parts[2]}"

# load in multi-session ROI data, binned to PSTH
raster_4d = nu.significant_trial_raster(target, alpha=0.05, bin_size_ms=20)

# get top-k value for ROI
topk_local = vst.fetch(f"{pth.OTHERS}/topk_vals.pkl")
with open(topk_local, "rb") as f:
    topk_vals = pickle.load(f)
top_k = int(topk_vals[roi_label]["k"])

print(f"Resolved ROI target: {target}")
print(f"Using top-k = {top_k}")
print(f"Raster shape after binning {raster_4d.shape}")

# shape (units, time, images)
raster_3d = np.nanmean(raster_4d, axis=3)
scores = tut.rank_images_by_response(raster_3d)

idx_topk = scores[:top_k]
trial_3d = raster_3d[:, :, idx_topk]
print(f"Trial averaged, all data {raster_3d.shape}")
print(f"Trial averaged, top-k shape {trial_3d.shape}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from scipy.linalg import subspace_angles

window_size = 100
step = 10
time_centers = np.arange(0, 450 - window_size, step)

rng = np.random.default_rng(0)
random_idxs = np.stack(
    [rng.choice(scores[top_k:], size=top_k, replace=False) for _ in range(100)],
    axis=0,
)

conditions = {
    'Top': scores[:100], # idx_topk,
    'Random (avg)': random_idxs,  # <-- now a stack
    'All': None,
}

results = {}

for label, image_idx in conditions.items():
    R_windows = [
        np.nanmean(raster_3d[:, t:t + window_size, :], axis=1).T
        for t in time_centers
    ]

    # ----- handle random averaging separately -----
    if label == 'Random (avg)':
        n_t = len(R_windows)
        angles_accum = np.zeros((n_t, n_t), dtype=float)

        for idx_set in image_idx:  # loop over random draws
            subspaces = []
            for R_t in R_windows:
                X_t = R_t[idx_set]
                A_t = PCA(n_components=2).fit(X_t).components_.T
                subspaces.append(A_t)

            angles_tt = np.zeros((n_t, n_t), dtype=float)
            for i in range(n_t):
                for j in range(n_t):
                    ang = subspace_angles(subspaces[i], subspaces[j])
                    angles_tt[i, j] = np.degrees(ang).mean()

            angles_accum += angles_tt

        angles_tt = angles_accum / image_idx.shape[0]

    # ----- standard cases -----
    else:
        # if image_idx is not None:
        #     to_index = rng.choice(image_idx, len(image_idx), replace=True)
        subspaces = []
        for R_t in R_windows:
            X_t = R_t if image_idx is None else R_t[image_idx]
            A_t = PCA(n_components=2).fit(X_t).components_.T
            subspaces.append(A_t)

        n_t = len(subspaces)
        angles_tt = np.zeros((n_t, n_t), dtype=float)
        for i in range(n_t):
            for j in range(n_t):
                ang = subspace_angles(subspaces[i], subspaces[j])
                angles_tt[i, j] = np.degrees(ang).mean()

    best, scores_cp = change_point_score(angles_tt)

    results[label] = {
        'angles_tt': angles_tt,
        'diag': diagonal_profile(angles_tt),
        'best': best,
        'scores': scores_cp,
    }

# ----- plotting -----
fig, axes = plt.subplots(1, len(results), figsize=(3 * len(results), 3), constrained_layout=True)

if len(results) == 1:
    axes = np.array([axes])

for row, (label, res) in enumerate(results.items()):
    ax = axes[row]
    sns.heatmap(res['angles_tt'], square=True, ax=ax)
    if row == 0:
        ax.set_title(f"{target} | {label}")
    else:
        ax.set_title(label)

    # sns.lineplot(res['diag'], ax=axes[row, 1])
    # axes[row, 1].set_title(f'{label}: diagonal profile')

plt.show()

for label, res in results.items():
    print(f'Best split {label.lower()}: {res["best"]}')

In [ ]:
cmap = sns.color_palette("viridis", as_cmap=True)
use_cbar = True  # or False

for label, res in results.items():
    R = res['angles_tt']

    fig = plt.figure(figsize=(2 + 0.5 * use_cbar, 2))
    if use_cbar:
        gs = fig.add_gridspec(1, 2, width_ratios=[20, 1])
        ax = fig.add_subplot(gs[0, 0])
        cax = fig.add_subplot(gs[0, 1])
    else:
        gs = fig.add_gridspec(1, 1)
        ax = fig.add_subplot(gs[0, 0])
        cax = None

    im = sns.heatmap(
        R,
        vmin=0,
        # vmax=1,  # change if your angle values exceed 1
        cmap=cmap,
        square=True,
        ax=ax,
        cbar=False,
    )

    if use_cbar:
        fig.colorbar(im.collections[0], cax=cax)#, ticks=[0, 0.5, 1])

    tick_positions = np.arange(0, R.shape[0] + 5, 5)
    tick_labels = np.arange(0, R.shape[0] + 5, 5) * 10
    ax.set_xticks(tick_positions)
    # ax.set_xticklabels(tick_labels)
    ax.set_yticks(tick_positions)
    ax.set_yticklabels(tick_labels)
    # ax.set_xlabel("Time (msec)")
    # ax.set_ylabel("Time")
    # ax.set_title(f"{target} | {label}")

    sns.despine(ax=ax, trim=True, offset=5)
    fig.tight_layout()
    heatmap_path = Path.home() / "Downloads" / f"{target}_{label}_pcangles.png"
    # plt.savefig(heatmap_path, dpi=300, bbox_inches='tight', transparent=True)
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from scipy.linalg import subspace_angles

window_size = 100
step = 10
time_centers = np.arange(0, 450 - window_size, step)

rng = np.random.default_rng(0)
n_boot = 100

random_idxs = np.stack(
    [rng.choice(scores[top_k:], size=top_k, replace=False) for _ in range(n_boot)],
    axis=0,
)

top_boot_idxs = np.stack(
    [rng.choice(idx_topk, size=len(idx_topk), replace=True) for _ in range(n_boot)],
    axis=0,
)

conditions = {
    'Top (avg)': top_boot_idxs,
    'Random (avg)': random_idxs,
    'All': None,
}

results = {}

for label, image_idx in conditions.items():
    R_windows = [
        np.nanmean(raster_3d[:, t:t + window_size, :], axis=1).T
        for t in time_centers
    ]

    # ----- handle averaged sampling cases -----
    if label in ('Top (avg)', 'Random (avg)'):
        n_t = len(R_windows)
        angles_accum = np.zeros((n_t, n_t), dtype=float)

        for idx_set in image_idx:
            subspaces = []
            for R_t in R_windows:
                X_t = R_t[idx_set]
                A_t = PCA(n_components=2).fit(X_t).components_.T
                subspaces.append(A_t)

            angles_tt = np.zeros((n_t, n_t), dtype=float)
            for i in range(n_t):
                for j in range(n_t):
                    ang = subspace_angles(subspaces[i], subspaces[j])
                    angles_tt[i, j] = np.degrees(ang).mean()

            angles_accum += angles_tt

        angles_tt = angles_accum / image_idx.shape[0]

    # ----- standard case -----
    else:
        subspaces = []
        for R_t in R_windows:
            X_t = R_t if image_idx is None else R_t[image_idx]
            A_t = PCA(n_components=2).fit(X_t).components_.T
            subspaces.append(A_t)

        n_t = len(subspaces)
        angles_tt = np.zeros((n_t, n_t), dtype=float)
        for i in range(n_t):
            for j in range(n_t):
                ang = subspace_angles(subspaces[i], subspaces[j])
                angles_tt[i, j] = np.degrees(ang).mean()

    best, scores_cp = change_point_score(angles_tt)

    results[label] = {
        'angles_tt': angles_tt,
        'diag': diagonal_profile(angles_tt),
        'best': best,
        'scores': scores_cp,
    }

# ----- plotting -----
fig, axes = plt.subplots(1, len(results), figsize=(3 * len(results), 3), constrained_layout=True)

if len(results) == 1:
    axes = np.array([axes])

for row, (label, res) in enumerate(results.items()):
    ax = axes[row]
    sns.heatmap(res['angles_tt'], square=True, ax=ax)
    if row == 0:
        ax.set_title(f"{target} | {label}")
    else:
        ax.set_title(label)

    # sns.lineplot(res['diag'], ax=axes[row, 1])
    # axes[row, 1].set_title(f'{label}: diagonal profile')

plt.show()

for label, res in results.items():
    print(f'Best split {label.lower()}: {res["best"]}')

In [ ]:
# parameters
seed = 0
n_random = 100
n_components = 3
t_start = 100
t_stop = 270

rng = np.random.default_rng(seed)
candidate_idxs = scores[top_k:]

# average over time window
resp = np.nanmean(raster_3d[:, t_start:t_stop, :], axis=1)  # (units, images)
R = resp.T                                                   # (images, units)

# subspaces for top-k and all images
A_top = PCA(n_components=n_components).fit(R[idx_topk]).components_.T
A_all = PCA(n_components=n_components).fit(R).components_.T

# direct comparisons
angles_top_all_deg = np.degrees(subspace_angles(A_top, A_all))
print(f"Top-k vs all-images angles: {angles_top_all_deg}")

# pre-sample random image sets
random_idxs = np.stack(
    [rng.choice(candidate_idxs, size=top_k, replace=False) for _ in range(n_random)],
    axis=0,
)

# top-vs-random
angles_top_rand_deg = np.zeros((n_random, n_components))
for i, idx_rand in enumerate(random_idxs):
    A_rand = PCA(n_components=n_components).fit(R[idx_rand]).components_.T
    angles_top_rand_deg[i] = np.degrees(subspace_angles(A_top, A_rand))

print(f"Top-vs-random mean angles: {angles_top_rand_deg.mean(axis=0)}")

# random-vs-random
angles_rand_rand_deg = np.zeros((n_random, n_components))
for i, idx_a in enumerate(random_idxs):
    idx_b = rng.choice(candidate_idxs, size=top_k, replace=False)
    A_rand_a = PCA(n_components=n_components).fit(R[idx_a]).components_.T
    A_rand_b = PCA(n_components=n_components).fit(R[idx_b]).components_.T
    angles_rand_rand_deg[i] = np.degrees(subspace_angles(A_rand_a, A_rand_b))

print(f"Random-vs-random mean angles: {angles_rand_rand_deg.mean(axis=0)}")

# all-images-vs-random
angles_all_rand_deg = np.zeros((n_random, n_components))
for i, idx_rand in enumerate(random_idxs):
    A_rand = PCA(n_components=n_components).fit(R[idx_rand]).components_.T
    angles_all_rand_deg[i] = np.degrees(subspace_angles(A_all, A_rand))

print(f"All-images-vs-random mean angles: {angles_all_rand_deg.mean(axis=0)}")

# overall scalar summaries
top_rand_mean = angles_top_rand_deg.mean(axis=1)
rand_rand_mean = angles_rand_rand_deg.mean(axis=1)
all_rand_mean = angles_all_rand_deg.mean(axis=1)
top_all_mean = angles_top_all_deg.mean()

# Top k is further from a random set than would be expected by change
print(f"\nTop-vs-random mean: {top_rand_mean.mean()}")
print(f"Random mean: {rand_rand_mean.mean()}")

# these 2 are similar
# my interpretation: top-k subspace is not further than All compared to Random
# or: global contains random + top-k
print(f"All-vs-random mean: {all_rand_mean.mean()}")
print(f"Top-vs-all mean: {top_all_mean}\n")

In [ ]:
df = pd.DataFrame({
    "vals": np.concatenate([
        top_rand_mean,
        rand_rand_mean,
        all_rand_mean,
        np.repeat(top_all_mean, len(top_rand_mean)),  # scalar → match length
    ]),
    "type": (
        ["top-rand"] * len(top_rand_mean) +
        ["rand-rand"] * len(rand_rand_mean) +
        ["all-rand"] * len(all_rand_mean) +
        ["top-all"] * len(top_rand_mean)
    )
})

fig,ax = plt.subplots(1,1)
sns.boxplot(data=df, x="type", y="vals", ax=ax)

ax.set_ylabel("Mean principal angle (deg)")
ax.set_xlabel("")

In [ ]:
subspaces = []
for t in time_centers:
    R_t = np.nanmean(raster_3d[:, t:t+window_size, :], axis=1).T
    R_top_t = R_t[random_idxs[7]]
    A_t = PCA(n_components=2).fit(R_top_t).components_.T
    subspaces.append(A_t)

n_t = len(subspaces)
angles_tt = np.zeros((n_t, n_t))
for i in range(n_t):
    for j in range(n_t):
        ang = subspace_angles(subspaces[i], subspaces[j])
        angles_tt[i, j] = np.degrees(ang).mean()

sns.heatmap(angles_tt, square=True)
plt.title("1 random")
plt.show()